# Analysis and Visualization of Complex Agro-Environmental Data
---
## Data Pre-processing: outlier analysis, data standardization and transformation

## 1. Outlier analysis

This refers to the analytical and graphical processes of identifying and examining data observations that in some way significantly differ from the rest of the dataset. This can be a deviation in terms of values that go much beyond the majority of values of a given variable (univariate outlier) or a combination of values that differ from a given pattern or trend in data (multivariate outlier).


Import modules:

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as sts
import seaborn as sns
import matplotlib.pyplot as plt
import statistics as stat

### 1.1. Example of the impact of outliers on descriptive statistics

In the following simple example, let's check the impact of including a single oulier in a dataset on the most common univariate descriptive statistics.

In [ ]:
# no outlier
v1 = [1,3,5,2,5,6,8,3,6] # list
v1 = pd.Series(v1) # pandas series
medianv1 = pd.Series({'median': v1.median()}) # compute median
statv1 = v1.describe() # compute summary statistics
table1 = pd.concat([medianv1, statv1]) # concatenate series

# add outlier (ex. 200)
v2 = [1,3,5,2,5,6,8,3,6,200] # list with outlier
v2 = pd.Series(v2) # pandas series
medianv2 = pd.Series({'median': v2.median()}) # compute median
statv2 = v2.describe() # compute summary statistics table
table2 = pd.concat([medianv2, statv2]) # concatenate series

pd.DataFrame({'No outlier': table1, 'With outlier': table2}) # combine resulting series into a single data frame.

As you can see, the mean and standard deviation changed drastically after adding the outlier. The median and percentiles did not change (in this example). In fact, these location measures are less sensitive to the presence of outliers.

### 1.2. Detection of outliers

Let's have a look at the dataset `winequality_red.csv` available [here](https://www.kaggle.com/datasets/sgus1318/winedata?resource=download) with 12 variables (columns) related with wine quality characteristics. 


In [ ]:
df_wine = pd.read_csv('winequality_red.csv')
df_wine.head()

In [ ]:
# Check the size of the dataset
len(df_wine)

#### 1.2.1 Tukey's Interquartile Range method

Let's first run a univariate scatterplot of the variable 'pH' as well as a boxplot:

In [ ]:
g = sns.JointGrid(data=df_wine, x=df_wine.index, y='pH') # create a JointGrid with the index of each observation on the x-axis and the pH values on the y-axis
g.plot(sns.scatterplot, sns.boxplot) # plot a scatterplot in the main axes and boxplots in the marginal axes
g.ax_marg_x.set_axis_off() # removes the marginal x-axis
g.ax_marg_y.set_axis_off() # removes the marginal y-axis
g.ax_marg_x.remove() # removes the marginal x boxplot (index of each observation)
g.set_axis_labels(xlabel='Observation index', ylabel='pH')
g.figure.set_figwidth(8) # to define the width to get a non-square figure with JointGrid 
g.figure.set_figheight(4) # to define the height to get a non-square figure with JointGrid 
plt.show()

This plot shows several outliers in the variable 'pH'. Boxplots are based on the concept of Interquartile Range (IQR) that is used in statistics to measure the statistical dispersion and data variability by dividing the dataset into quartiles. The first, second and fourth quartiles divides the data into four intervals of equal probability: <Q1; Q1-Q2; Q2-Q3; >Q3.

IQR is the difference between the third quartile and the first quartile (IQR = Q3 -Q1). Outliers using this criteria are defined as ***the observations that are below Q1 − 1.5 x IQR (i.e. defining the lower whisker), or above Q3 + 1.5 x IQR (i.e. defining the upper whisker)***.

We can run some code to try to count and detect these outliers:

In [ ]:
# function to compute the IQR (upper and lower whisker) and count outliers

def out_iqr(df , column): # define function with arguments for the data frame and the column name to analyze
    global loweriqr,upperiqr # to make the lower and upper whisker values available outside the function
    q25, q75 = np.quantile(df[column], 0.25), np.quantile(df[column], 0.75) # compute the 25th and 75th percentiles
    # calculate the IQR
    iqr = q75 - q25
    # calculate the outlier cutoff
    cut_off = iqr * 1.5
    # calculate the lower and upper bound value
    loweriqr, upperiqr = q25 - cut_off, q75 + cut_off
    # print results
    print('IQR = ',iqr)
    print('lower whisker = ', round(loweriqr, 3))
    print('upper whisker = ', round(upperiqr, 3))
    # Calculate the number of records below and above lower and above bound value respectively
    df1 = df[df[column] > upperiqr]
    df2 = df[df[column] < loweriqr]
    return print('Total number of outliers = ', df1.shape[0]+ df2.shape[0])

In [ ]:
# Get the IQR limits ad the number of outliers
out_iqr(df_wine,'pH')

In [ ]:
# Produce histogram with areas defining outlier 
plt.figure(figsize = (10,6)) # to define the size of the figure
sns.set_style('darkgrid') # set the background to grey with white tckmarks (ggplot default style)
sns.histplot(df_wine.pH, 
    color='skyblue',
    edgecolor='none' # to remove the black border around the bars
    )
plt.axvspan(xmin= df_wine.pH.min(), xmax = loweriqr, alpha=0.2, color='red') # lower was defined by the function 'out_iqr' produced above
plt.axvspan(xmin = upperiqr,xmax= df_wine.pH.max(),alpha=0.2, color='red') # upper was defined by the function 'out_iqr' produced above
plt.show()

In [ ]:
# To go back to matplotlib default style run:
plt.style.use('default')

In [ ]:
# Get the position (index) of the outliers in the dataset
outlier_indices = np.where((df_wine['pH'] > upperiqr) | (df_wine['pH'] < loweriqr)) # get the index of the records with pH values above the upper whisker or below the lower whisker
print(outlier_indices)
print(np.size(outlier_indices)) # get the number of outliers

In [ ]:
# add a column to identify the IQR-based outliers
df_wine['Outlier_iqr'] = (df_wine['pH'] < loweriqr) | (df_wine['pH'] > upperiqr) # create a new column with boolean values (True for outliers and False for non-outliers)
pd.crosstab(df_wine['Outlier_iqr'], columns='count') # count the number of outliers and non-outliers

In [ ]:
# Visualize the univariate ouliers of the dataset
g = sns.JointGrid(data=df_wine, x=df_wine.index, y='pH', hue='Outlier_iqr') # create a JointGrid with the index of each observation on the x-axis and the pH values on the y-axis, with different colors for outliers and non-outliers
g.plot(sns.scatterplot, sns.boxplot) # plot a scatterplot in the main axes and boxplots in the marginal axes, with different colors for outliers and non-outliers
g.ax_marg_x.set_axis_off() # removes the marginal x-axis
g.ax_marg_y.set_axis_off() # removes the marginal y-axis
g.ax_marg_x.remove() # removes the marginal x boxplot (index of each observation)
g.set_axis_labels(xlabel='Observation index', ylabel='pH')
g.figure.set_figwidth(8) # to define the width to get a non-square figure with JointGrid 
g.figure.set_figheight(4) # to define the height to get a non-square figure with JointGrid 
plt.show()

In [ ]:
# Remove outliers and create new data frame
df_new = df_wine.loc[(df_wine['pH'] < upperiqr) & (df_wine['pH'] > loweriqr)]
len(df_new)

In [ ]:
# Check the impact on univariate summary statstics of removing outliers
# original dataset:
median_pH = pd.Series({'median': df_wine['pH'].median()})
stat_pH = df_wine['pH'].describe()
table1 = pd.concat([median_pH, stat_pH])
# no outliers:
median_pH2 = pd.Series({'median': df_new['pH'].median()})
stat_pH2 = df_new['pH'].describe()
table2 = pd.concat([median_pH2, stat_pH2])

pd.DataFrame({'Original': table1, 'No outliers': table2})

#### 1.2.2 Standard deviation method

This method uses the standard deviation of the sample as a cut-off for identifying outliers and applies <u>**only to data with Gaussian-like distributions**</u>.

As we saw, if a data distribution is normal then approximately:

* 68% of the data values lie within one standard deviation of the mean
* 95% are within two standard deviations
* 99.7% lie within three standard deviations.

Outliers may be defined acording to a cutoff value corresponding to <u>**2 times stdev or 3 times stdev**</u>.
The code below follows an approach similar to the previous method:

In [ ]:
# function to detect values below mean - 3xstdev or above mean + 3xstdev, that will define the outliers

def out_std(df, column): # define function with arguments for the data frame and the column name to analyze
    global lowersd, uppersd # to make the lower and upper bound values available outside the function
    # calculate the mean and standard deviation of the data frame
    data_mean, data_std = df[column].mean(), df[column].std()
    # calculate the cutoff value
    cut_off = data_std * 3
    # calculate the lower and upper bound value
    lowersd, uppersd = data_mean - cut_off, data_mean + cut_off
    print('The lower bound value is', round(lowersd, 3))
    print('The upper bound value is', round(uppersd, 3))
    # Calculate the number of records below and above lower and above bound value respectively
    df1 = df[df[column] > uppersd]
    df2 = df[df[column] < lowersd]
    return print('Total number of outliers are', df1.shape[0]+ df2.shape[0])

In [ ]:
# Get the 3xSD limits ad the number of outliers
out_std(df_wine,'pH')

In [ ]:
# add a column to identify the SD-based outliers
df_wine['Outlier_sd'] = (df_wine['pH'] < lowersd) | (df_wine['pH'] > uppersd)
pd.crosstab(df_wine['Outlier_sd'], columns='count')

In [ ]:
# Visualize the univariate ouliers of the dataset
g = sns.JointGrid(data=df_wine, x=df_wine.index, y='pH', hue='Outlier_sd')
g.plot(sns.scatterplot, sns.boxplot)
g.ax_marg_x.set_axis_off() # removes the marginal x-axis
g.ax_marg_y.set_axis_off() # removes the marginal y-axis
g.ax_marg_x.remove() # removes the marginal x boxplot (index of each observation)
g.set_axis_labels(xlabel='Observation index', ylabel='pH')
g.fig.set_figwidth(8) # to define the width to get a non-square figure with JointGrid 
g.fig.set_figheight(4) # to define the height to get a non-square figure with JointGrid 
plt.show()

This method tends to select fewer outliers. Because outliers increase the standard deviation, the method **may fail to detect outliers**.

#### 1.2.3 Z-score method

Z-score describes the position of a raw score in terms of its distance from the mean, when measured in standard deviation units. This method also <u>**assumes a Gaussian distribution of the data**</u>. The outliers will be defined as the data points that are located in the tails of the distribution.

`Z_score = (Xi - mean) / standard deviation`

where:
* Xi - observation
* 'mean' - mean of all Xi 
* 'standard deviation' - standard deviation of all Xi

An outlier is then a normalized data point which has an absolute value greater than the <u>**Z-score threshold**</u> (Zthr). That is:
|Z_score| > Zthr

Commonly used Z-score threshold values are <u>**2.5, 3.0 or 3.5</u>. Here we will be using 3.5

Following a similar approach to the previous methods:

In [ ]:
# Function to find the lower and the upper cutoff values using the Z-Score method

def out_zscore(data):
    global outliers,zscore
    outliers = []
    zscore = []
    threshold = 3.5
    mean = np.mean(data)
    std = np.std(data)
    for i in data:
        z_score= (i - mean)/std 
        zscore.append(z_score)
        if np.abs(z_score) > threshold:
            outliers.append(i)
    return print("Total number of outliers = ",len(outliers))


In [ ]:
# Get the number of outliers
out_zscore(df_wine['pH'])

In [ ]:
# Create a data frame with the Z-Score values
df_pH = pd.DataFrame(zscore, columns=['pH'])
df_pH

In [ ]:
# add a column to identify the SD-based outliers
df_wine['Outlier_zscore'] = (df_pH['pH'] < -3.5) | (df_pH['pH'] > 3.5)
pd.crosstab(df_wine['Outlier_zscore'], columns='count')

In [ ]:
# Visualize the univariate ouliers of the dataset
g = sns.JointGrid(data=df_wine, x=df_wine.index, y='pH', hue='Outlier_zscore')
g.plot(sns.scatterplot, sns.boxplot)
g.ax_marg_x.set_axis_off() # removes the marginal x-axis
g.ax_marg_y.set_axis_off() # removes the marginal y-axis
g.ax_marg_x.remove() # removes the marginal x boxplot (index of each observation)
g.set_axis_labels(xlabel='Observation index', ylabel='pH')
g.fig.set_figwidth(8) # to define the width to get a non-square figure with JointGrid 
g.fig.set_figheight(4) # to define the height to get a non-square figure with JointGrid 
plt.show()

#### 1.2.4 Isolation Forest

Isolation forest is based recursive partitioning of the dataset using a set of <u>**decision trees**</u> and provides an <u>**anomaly score**</u> that measures  how isolated the point is in the structure found. The anomaly score is then used to detect outliers. The number of splittings required to isolate a sample is equivalent to the path length from the root node to the terminating node. This method is based on the assumption that <u>**it requires fewer splits to isolate an outlier than it does to isolate a non-outlier**</u>, i.e. an outlier has a lower isolation number in comparison to a non-outlier observation. 

An observation is therefore defined as an outlier  <u>**if its isolation number is lower than a given threshold**</u>. The threshold is defined based on the estimated percentage of outliers in the data, which is the starting point of this outlier detection algorithm.

Isolation Forest does <u>**not assume normal distribution**</u> and is <u>**able to detect outliers at a multi-dimensional level**</u>. Moreover, the algorithm has a linear time complexity with a low constant and a low memory requirement which makes it computationally efficient. Hence, it scales well to large data sets.

The method is implemented in the scikit-learn module: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html

In the following code, examples of how to identify ***univariate, bivariate and multivariate outliers*** are provided.

In [ ]:
#Import necessary libraries
from sklearn.ensemble import IsolationForest

##### Run a univariate isolation forest (to detect univariate outliers)

Let's run a univariate isolation forest for 'pH'

In [ ]:
model = IsolationForest(n_estimators = 150, contamination='auto') # define the model (150 isolation trees, default: only 1 variable (feature) of the dataset)
model.fit(df_wine['pH'].values.reshape(-1,1)) # fit the model (univariate: 'pH')
scores = model.decision_function(df_wine['pH'].values.reshape(-1,1)) #  extract the anomaly scores (mean anomaly scores of each tree) - vary between -1 and 1
anomaly = model.predict(df_wine['pH'].values.reshape(-1,1)) # extract an anomaly (outlier) identifier (-1)
df_wine['scores'] = scores # add a new column to the database with the anomaly scores
df_wine['anomaly'] = anomaly # add a new column to the database with the anomaly identifier (1/-1)

pd.crosstab(df_wine['anomaly'], columns='count') # count the number of outliers

Visualize the univariate ouliers of the dataset

In [ ]:
g = sns.JointGrid(data=df_wine, x=df_wine.index, y='pH', hue='anomaly')
g.plot(sns.scatterplot, sns.boxplot)
g.ax_marg_x.set_axis_off() # removes the marginal x-axis
g.ax_marg_y.set_axis_off() # removes the marginal y-axis
g.ax_marg_x.remove() # removes the marginal x boxplot (index of each observation)
g.set_axis_labels(xlabel='Observation index', ylabel='pH')
g.fig.set_figwidth(8) # to define the width to get a non-square figure with JointGrid 
g.fig.set_figheight(4) # to define the height to get a non-square figure with JointGrid 
plt.show()

##### Run a bivariate isolation forest

Let's now run a bivariate isolation forest for'pH' and 'citric acid'

In [ ]:
model = IsolationForest(n_estimators = 150, contamination='auto', max_features=2) # define the model (150 isolation trees, default: only 1 variable (feature) of the dataset)
model.fit(df_wine[['citric acid', 'pH']].values) # fit the model (univariate: 'pH')
scores = model.decision_function(df_wine[['citric acid', 'pH']].values) #  extract the anomaly scores (mean anomaly scores of each tree)
anomaly = model.predict(df_wine[['citric acid', 'pH']].values) # extract an anomaly (outlier) identifier (-1)
df_wine['scores'] = scores # add a new column to the database with the anomaly scores
df_wine['anomaly'] = anomaly # add a new column to the database with the anomaly identifier (1/-1)
pd.crosstab(df_wine['anomaly'], columns='count') # count the number of outliers

Visualize the bivariate ouliers of the dataset

In [ ]:
g = sns.JointGrid(data=df_wine, x='citric acid', y='pH', hue='anomaly') 
g.plot(sns.scatterplot, sns.histplot) # to plot both a scatter plot and a boxplot for each variable
g.ax_marg_x.set_axis_off() # removes the marginal x-axis
g.ax_marg_y.set_axis_off() # removes the marginal y-axis
g.fig.set_figwidth(7) # to define the width to get a non-square figure with JointGrid 
g.fig.set_figheight(4) # to define the height to get a non-square figure with JointGrid 
plt.show()

Alternative using continuous anomaly scores instead (may take a while).

In [ ]:
g = sns.JointGrid(data=df_wine, x='citric acid', y='pH', hue='scores') 
g.plot(sns.scatterplot, sns.histplot) # to plot both a scatter plot and a boxplot for each variable
g.ax_marg_x.set_axis_off() # removes the marginal x-axis
g.ax_marg_y.set_axis_off() # removes the marginal y-axis
g.figure.set_figwidth(7) # to define the width to get a non-square figure with JointGrid 
g.figure.set_figheight(4) # to define the height to get a non-square figure with JointGrid 
plt.show()

##### Run a multivariate isolation forest

Let's now run a multivariate isolation forest for all variables in the dataset (including categorical)

In [ ]:
model = IsolationForest(n_estimators = 150, contamination='auto', max_features=12) # define the model (150 isolation trees, all 12 variables in the dataset)
model.fit(df_wine.values) # fit the model to the data for all 12 variables in the dataset
scores = model.decision_function(df_wine.values) # extract the anomaly scores (mean anomaly scores of each tree)
anomaly = model.predict(df_wine.values) # to extract an anomaly (outlier) identifier (-1)
df_wine['scores'] = scores # add a new column to the database with the anomaly scores
df_wine['anomaly'] = anomaly # add a new column to the database with the anomaly identifier(1/-1)
pd.crosstab(df_wine['anomaly'], columns='count') # count the number of outliers

Visualize the multivariate ouliers of the dataset using a univariate plot for 'pH' (just as an example).

NOTE: It is dificult to visualize outliers for more than 2 dimensions (maybe a biplot of a Principal Component Analysis would help).

In [ ]:
g = sns.JointGrid(data=df_wine, x=df_wine.index, y='pH', hue='anomaly')
g.plot(sns.scatterplot, sns.boxplot)
g.ax_marg_x.set_axis_off() # removes the marginal x-axis
g.ax_marg_y.set_axis_off() # removes the marginal y-axis
g.ax_marg_x.remove() # removes the marginal x boxplot (index of each observation)
g.set_axis_labels(xlabel='Observation index', ylabel='pH')
g.figure.set_figwidth(8) # to define the width to get a non-square figure with JointGrid 
g.figure.set_figheight(4) # to define the height to get a non-square figure with JointGrid 
plt.show()

### 1.2.5 Conclusion

The different detection methods differ greatly leading to marked differences in outlier detection. It is important to detect outliers by finding consensus among different methods. After detecting outliers, it important to assess whether they correspond to real data as well as to evaluate their impact on the analytical methods we intend to use. Based on this assessment, we may use different approaches to deal with them.

There are many other methods for outlier detection processes that you may search for, such as:
- DBSCAN (Density-Based Spatial Clustering of Applications with Noise)
- Local Outlier Factor Method (LOF):
- Elliptic Envelope
- One-Class Support Vector Machines
- Robust Random Cut Forest

### 2. Variable standardization and normalization

Variable standardization is used in many situations, namely when analyses involve variables measured in different units. It provides methods of rescaling variables without changing the probability distributions of the original data. It is a requirement and a common practice when certain statistical approaches and modelling algorithms are applied to data. For example, in regression-based methods, it allows to directly compare the effect sizes of variables that are measured in very different units; in some multivariate statistical approaches, it ensures that variables measured in different units will have equal weights in the analysis; when applying certain modelling algorithms such as Generalized Linar Mixed Models, a previous data standardization is a requirement.

The most commonly used standardization method involves centering and scaling operations (**Z-score standardization**), i.e., each observation  is subtracted by the mean and divided by the standard deviation, so that the new standardized variables has mean = 0 and the standard deviation = 1. After this standardization transformation, the transformed data is often called **Z-score**.

Standardizing variables is quite straightforward in python. Even so, there are some modules that implement variable standardization functions, such as the `zscore()` of `statsmodels` or the `StandardScaler()` function of `scikit-learn`.

##### 2.1 Standardize single variable

In [ ]:
# Create variable
var = sts.norm.rvs(scale=50, loc=150, size=1000)
sns.histplot(var)
plt.show()

In [ ]:
# Standardize
varstd = (var-var.mean())/var.std()
sns.histplot(varstd)
plt.show()

In [ ]:
# Alternative 2

varstd2 = sts.zscore(var) # zscore function that will standardize the variable
sns.histplot(varstd2)
plt.show()

In [ ]:
# Alternative 3
# Import Z-Score Standard Scaler from the Sklearn package
from sklearn.preprocessing import StandardScaler
scale = StandardScaler() # create an instance of the StandardScaler class
var2 = pd.DataFrame(var) # convert the variable to a DataFrame (it is required by the fit_transform function)
varstd3 = scale.fit_transform(var2) # standardize the variable using the fit_transform function of the StandardScaler class
sns.histplot(varstd3)
plt.show()

##### 2.2 Standardize several variables at once

In [ ]:
# Simulate data frame
df3 = pd.DataFrame({'x1': [5, 7, 7, 9, 12, 9, 9, 4],
                   'x2': [11, 8, 10, 6, 6, 5, 9, 12],
                   'x3': [2, 2, 3, 2, 5, 5, 7, 9]})

In [ ]:
df4 = df3.copy() # create a copy of df3 to apply transformations

# standardize the values for each predictor variable - replace values in df by standardized ones
df4[['x1', 'x2', 'x3']] = (df3-df3.mean())/df3.std()

#view new data frame
df4

In [ ]:
# mean of each predictor variable column
df4[['x1', 'x2', 'x3']].mean()

In [ ]:
# standard deviation of each predictor variable column
df4[['x1', 'x2', 'x3']].std()

##### 2.2 Normalize data using min-max scaler

It normalizes the data into a range between 0 and 1 based on the formula: (x - x<sub>min</sub>)/(x<sub>max</sub> - x<sub>min</sub>)

In [ ]:
# define predictor variable columns
df5 = df3.copy() # create a copy of df3 to apply transformations

# standardize the values for each predictor variable - replace values in df by stadardized ones
df5[['x1', 'x2', 'x3']] = (df5-df5.min())/(df5.max()-df5.min())
df5

### 3. Variable transformation

Variable transformation is a very commonly used procedure to prevent some potential statistical problems, especially when using parametric hypothesis testing such as t-test or regression-based methods. It is especially used to tranform variables with very skewed distributions to become more belly shaped. 

**Logarithmic transformation** is one of the most commonly used. A more general and complex transformations are the **Box-Cox transformations**, which involves the estimation of a parameter (lambda) to approximate as much as possible the variable to a normal distribution. For **proportions**, angular transformations such as the **arcsin** are more appropriate.

In [ ]:
# Simulate right skewed variable
var = sts.expon.rvs(scale=1, loc=0, size=1000)
sns.histplot(var)
plt.show()

In [ ]:
# Logarithmic transformation
varlog = np.log10(var)
sns.histplot(varlog)
plt.show()

In [ ]:
# Square-root transformation
varsqrt = np.sqrt(var)
sns.histplot(varsqrt)
plt.show()

In [ ]:
# Cube-root transformation
varcbrt = np.cbrt(var)
sns.histplot(varcbrt)
plt.show()

In [ ]:
# Box-cox transformation
from scipy import stats

fitted_data, fitted_lambda = stats.boxcox(var)

# creating axes to draw plots
fig, ax = plt.subplots(1, 2)
 
# plotting the original data(non-normal) and
# fitted data (normal)
sns.histplot(var, kde = True,
            label = "Original", ax = ax[0]).legend(loc = "upper right")
 
sns.histplot(fitted_data, kde = True,
            label = "Transformed", ax = ax[1]).legend(loc = "upper right")

# rescaling the subplots
fig.set_figheight(5)
fig.set_figwidth(10)
 
print(f"Lambda value used for Transformation: {round(fitted_lambda, 3)}")

In [ ]:
# Transform the values for each predictor variable in a table - replace values in df6 (see above) by transformed ones
df6 = df3.copy()
df6[['x1', 'x2', 'x3']] = np.log10(df6[['x1', 'x2', 'x3']])
df6
# Alternative:
# df3[['x1', 'x2', 'x3']] = df3[['x1', 'x2', 'x3']].apply(np.log10)


In [ ]:
# Now with a left skewed variable?
from scipy.stats import skewnorm

var2 = skewnorm.rvs(a = -10,loc=100, size=10000)

sns.histplot(var2)
plt.show()

In [ ]:
# Box-cox transformation

fitted_data, fitted_lambda = stats.boxcox(var2)

# creating axes to draw plots
fig, ax = plt.subplots(1, 2)
 
# plotting the original data(non-normal) and
# fitted data (normal)
sns.histplot(var2, kde = True,
            label = "Original", ax = ax[0]).legend(loc = "upper right")
 
sns.histplot(fitted_data, kde = True,
            label = "Transformed", ax = ax[1]).legend(loc = "upper right")

# rescaling the subplots
fig.set_figheight(5)
fig.set_figwidth(10)
 
print(f"Lambda value used for Transformation: {round(fitted_lambda, 3)}")

In [ ]:
# The fitted_data results from using the following expression:
varlog = (var2**fitted_lambda-1)/fitted_lambda
sns.histplot(varlog)
plt.show()

### References

A walkthrough of Univariate Anomaly Detection in Python https://www.analyticsvidhya.com/blog/2021/06/univariate-anomaly-detection-a-walkthrough-in-python/

How to Standardize Data in Python (With Examples), https://www.statology.org/standardize-data-python/

IsolationForest https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html

Normalize a Pandas Column or Dataframe (w/ Pandas or sklearn), https://datagy.io/pandas-normalize-column/

Outlier Detection Techniques: Simplified https://www.kaggle.com/code/rpsuraj/outlier-detection-techniques-simplified

Python | Box-Cox Transformation, https://www.geeksforgeeks.org/box-cox-transformation-using-python/

